In [1]:
try:
    !pip install -q \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-chroma \
    groq
    print("Packages installed successfully")
except Exception as e:
    print(f"Error installing packages: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170

Mount Google Drive

In [2]:
from google.colab import drive

try:
    drive.mount('/content/drive')
    print("Drive mounted successfully")
except Exception as e:
    print("Error:", e)

Mounted at /content/drive
Drive mounted successfully


In [3]:
import os
from google.colab import userdata

try:

    os.environ["GOOGLE_API_KEY"] = (
        userdata.get("Gemini_key")
    )

    GROQ_API_KEY = userdata.get(
        "Groq_Key"
    )

    print("API Keys Loaded Successfully")

except Exception as e:

    print("Error:", e)

API Keys Loaded Successfully


In [4]:
try:
    from langchain_chroma import Chroma
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from groq import Groq
    print("Imported successfully")
except ImportError as e:
    print(f"Import Error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

Imported successfully


Embedddings

In [5]:
try:
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )
    print("Embeddings ready")
except Exception as e:
    print("Error:", e)

Embeddings ready


Connect to Existing ChromaDB

In [6]:
CHROMA_DIR = "/content/drive/MyDrive/RAG_Internship/chroma_db"

try:
    db = Chroma(
        collection_name="wiki_rag",
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR
    )
    print("Documents:", db._collection.count())
except Exception as e:
    print("Error:", e)

Documents: 489


Retriver

In [7]:
try:
    retriever = db.as_retriever(search_kwargs={"k": 5})
    print("Retriever ready")
except Exception as e:
    print("Error:", e)

Retriever ready


In [8]:
try:
    query = "What is machine learning?"
    results = retriever.invoke(query)
    print(f"Retrieved {len(results)} chunks")
except NameError:
    print("Retriever not initialized")
except Exception as e:
    print(f"Error retrieving documents: {e}")

Retrieved 5 chunks


In [9]:
try:
    if 'results' in locals() and results:
        for i, doc in enumerate(results, 1):
            print("=" * 50)
            print(f"Chunk {i}")
            print(doc.page_content[:500])
    else:
        print("No results to display")
except Exception as e:
    print(f"Error displaying chunks: {e}")

Chunk 1
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose the foundations o
Chunk 2
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose 

Answer Generation

In [10]:
try:
    client = Groq(api_key=GROQ_API_KEY)
    print("Groq initialized Successfully")
except Exception as e:
    print("Error:", e)

Groq initialized Successfully


Create RAG Function

In [11]:
def ask_question(query):

    try:

        docs = retriever.invoke(query)

        if len(docs) == 0:

            return (
                "No relevant documents found."
            )

        context = "\n\n".join(
            [
                doc.page_content
                for doc in docs
            ]
        )

        prompt = f"""
You are a helpful assistant.

Answer ONLY using the provided context.

If the answer is not present
in the context, say:

'I could not find that information
in the documents.'

Context:
{context}

Question:
{query}
"""

        response = (
            client.chat.completions.create(
                model=
                "llama-3.3-70b-versatile",
                messages=[
                    {
                        "role":"user",
                        "content":prompt
                    }
                ]
            )
        )

        return (
            response
            .choices[0]
            .message
            .content
        )

    except Exception as e:

        return f"Error: {e}"

Queries

In [12]:
try:
    result = ask_question("What is deep learning?")
    print(result)
except Exception as e:
    print(f" Error: {e}")

Deep learning refers to a class of machine learning algorithms in which a hierarchy of layers is used to transform input data into a progressively more abstract and composite representation. It utilizes multilayered neural networks to perform tasks such as classification, regression, and representation learning, and can learn which features to optimally place at which level on its own.


In [13]:
try:
    result = ask_question("What is computer vision?")
    print(result)
except Exception as e:
    print(f" Error: {e}")

Computer vision is an interdisciplinary field that deals with how computers can be made to gain high-level understanding from digital images or videos. It seeks to automate tasks that the human visual system can do, and involves the development of a theoretical and algorithmic basis to achieve automatic visual understanding. It is concerned with the automatic extraction, analysis, and understanding of useful information from a single image or a sequence of images.


In [15]:
try:
    result = ask_question("Who is Mahathma Ghandi?")
    print(result)
except Exception as e:
    print(f" Error: {e}")

I could not find that information in the documents.


# Notebook Documentation

## Overview
This notebook implements a RAG (Retrieval-Augmented Generation) query pipeline. It retrieves relevant document chunks from ChromaDB and generates answers using Groq's Llama 3.3 model.

## Dependencies
- langchain, langchain-chroma, langchain-google-genai
- groq

## Core Concepts

### Retrieval-Augmented Generation (RAG)
RAG combines information retrieval with text generation. The process:
1. **Retrieve**: Find relevant documents/chunks from vector database
2. **Augment**: Add retrieved context to the prompt
3. **Generate**: LLM produces answer grounded in retrieved context

### Retriever
A component that fetches relevant chunks based on query semantics:
- Uses **cosine similarity** to find nearest vectors
- Returns top-k most relevant chunks (k=5 in this notebook)
- Performs semantic search (meaning-based, not keyword-based)

### Context Window
The retrieved chunks provide context for LLM:
- Prevents hallucination by grounding answers
- Limits response to available information
- Enables source attribution

### Similarity Search
Vector database compares query embedding with stored embeddings:
- **Cosine Similarity**: Measures angle between vectors (1 = identical)
- **Top-k Retrieval**: Returns k most similar chunks
- **Semantic Matching**: Finds related concepts, not just exact words

## Data Source

### Prerequisite
Must run `Ingestion.ipynb` first to create:
- ChromaDB with 289 chunks from Wikipedia articles
- Vector embeddings (3072 dimensions) stored in Google Drive

## Configuration

### API Keys Required

| Secret Name | Purpose |
|-------------|---------|
| `Gemini_key` | Generate embeddings for queries |
| `Groq_Key` | Access Llama 3.3 model |


## Code Structure

| Cell(s) | Function |
|---------|----------|
| 1 | Install required dependencies |
| 2 | Mount Google Drive |
| 3 | Load API keys (Gemini, Groq) |
| 4 | Import libraries |
| 5 | Initialize Gemini embeddings |
| 6 | Connect to existing ChromaDB |
| 7 | Create retriever |
| 8-9 | Test retriever & display chunks |
| 10 | Initialize Groq client |
| 11 | Create RAG function |
| 12 | Query: Deep Learning |
| 13 | Query: Computer Vision |
| 14 | Query: Alan Turing |